# ⚡ LNG Deal Feasibility — Extended (Option A)
### Credit · IFRS 9 · Netback · Vessel · Approval

This notebook **extends** [`lng_feasibility.ipynb`](lng_feasibility.ipynb).

**Workflow:**
1. Load a deal exported from `lng_feasibility.ipynb` (JSON) — *or* enter manually
2. Run credit & counterparty checks
3. Model IFRS 9 hedge accounting (OCI vs P&L)
4. Compare netback across multiple destinations
5. Select and schedule a vessel
6. Simulate the deal approval workflow
7. Produce an approval-ready deal summary

---
> 💡 **To link notebooks:** Run Section 8 in `lng_feasibility.ipynb` first to export `lng_deal_YYYYMMDD.json`, then load it in Cell 2 below.
---

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 0 — ENVIRONMENT SETUP (Colab + Local Jupyter)
# ─────────────────────────────────────────────────────────────────────────────

import subprocess, sys, shutil

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

for pkg in ['ipywidgets', 'scipy']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

if not IN_COLAB:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                               'widgetsnbextension', 'jupyterlab-widgets'])
        if shutil.which('jupyter'):
            subprocess.run(['jupyter', 'nbextension', 'enable', '--py',
                            'widgetsnbextension', '--sys-prefix'],
                           capture_output=True)
    except Exception:
        pass

print('✅  Environment ready.', '(Colab)' if IN_COLAB else '(Local Jupyter)')
if not IN_COLAB:
    print('   ℹ️  If sliders don\'t appear, restart the notebook server after this first run.')

In [ ]:
# ── CELL 1: Imports & styling ─────────────────────────────────────────────────
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.gridspec import GridSpec
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from datetime import date, timedelta
import json, os, glob

%matplotlib inline
plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor':   '#30363d', 'axes.labelcolor': '#e6edf3',
    'xtick.color': '#8b949e',      'ytick.color': '#8b949e',
    'text.color':  '#e6edf3',      'grid.color':  '#21262d',
    'grid.linestyle': '--',        'grid.alpha':  0.5,
    'font.size': 10, 'axes.titlesize': 11,
    'axes.titleweight': 'bold', 'figure.dpi': 110,
})
C = {
    'gain':'#3fb950','loss':'#f85149','warn':'#d29922',
    'blue':'#58a6ff','purple':'#bc8cff','teal':'#39d353',
    'bg':'#0d1117','card':'#161b22','border':'#30363d',
    'text':'#e6edf3','muted':'#8b949e',
}
print('✅  Option A loaded. Proceed to Cell 2 to load your deal.')


---
## 📂 Section 1 — Load Deal from `lng_feasibility.ipynb`
*Pick the exported JSON or enter key values manually.*

In [ ]:
# ── CELL 2: Load deal JSON ────────────────────────────────────────────────────
style  = {'description_width': '200px'}
layout = widgets.Layout(width='500px')

# Auto-detect exported JSON files
def refresh_json_list():
    found = sorted(glob.glob('lng_deal_*.json'), reverse=True)
    return found if found else ['No JSON found — enter manually']

json_opts = refresh_json_list()

w_json_file = widgets.Dropdown(
    options=json_opts, value=json_opts[0],
    description='Load Deal JSON:',
    style=style, layout=layout
)
w_load_btn = widgets.Button(
    description='📂  Load JSON',
    button_style='info',
    layout=widgets.Layout(width='160px', height='36px')
)
load_out = widgets.Output()

# Deal state — populated from JSON or manual entry
deal_state = {}

def load_json(_=None):
    global deal_state
    fname = w_json_file.value
    with load_out:
        clear_output(wait=True)
        if os.path.exists(fname):
            with open(fname) as f:
                deal_state = json.load(f)
            p = deal_state.get('pricing', {})
            v = deal_state.get('volumes', {})
            pnl = deal_state.get('pnl', {})
            print(f"✅  Loaded: {fname}")
            print(f"   Verdict:       {deal_state.get('meta',{}).get('verdict','?')}")
            print(f"   Net Delivered: {v.get('net_delivered_mmbtu',0):,.0f} MMBtu")
            print(f"   Buy Price:     ${p.get('buy_price_mmbtu',0):.4f}/MMBtu")
            print(f"   Sell Price:    ${p.get('sell_price_mmbtu',0):.4f}/MMBtu")
            print(f"   Base Net P&L:  ${pnl.get('net_profit',0):+,.0f}")
            # Populate manual overrides
            w_net_mmbtu.value     = float(v.get('net_delivered_mmbtu', 1362090))
            w_buy_px.value        = float(p.get('buy_price_mmbtu', 13.28))
            w_sell_px.value       = float(p.get('sell_price_mmbtu', 14.20))
            w_base_pnl.value      = float(pnl.get('net_profit', 0))
            w_total_costs.value   = float(pnl.get('total_freight', 0) +
                                          pnl.get('regas_storage', 0) +
                                          pnl.get('bog_expense', 0) +
                                          pnl.get('other_costs', 0))
        else:
            print(f"⚠️  File not found: {fname}. Use manual inputs below.")

# ── Upload support (Colab + Local Jupyter) ────────────────────────────────────
upload_out = widgets.Output()

if IN_COLAB:
    w_upload_btn = widgets.Button(
        description='📤  Upload JSON file',
        button_style='warning',
        layout=widgets.Layout(width='200px', height='36px')
    )
    def _colab_upload(_=None):
        with upload_out:
            clear_output(wait=True)
            uploaded = colab_files.upload()
            if uploaded:
                for fname in uploaded:
                    print(f"✅  Uploaded: {fname}")
                new_opts = refresh_json_list()
                w_json_file.options = new_opts
                w_json_file.value = new_opts[0]
                load_json()
    w_upload_btn.on_click(_colab_upload)
else:
    w_upload_btn = widgets.FileUpload(
        accept='.json', multiple=False,
        description='📤  Upload JSON',
        layout=widgets.Layout(width='200px')
    )
    def _local_upload(change):
        with upload_out:
            clear_output(wait=True)
            for fname, file_info in w_upload_btn.value.items():
                content = file_info['content']
                with open(fname, 'wb') as f:
                    f.write(content)
                print(f"✅  Uploaded: {fname}")
            new_opts = refresh_json_list()
            w_json_file.options = new_opts
            w_json_file.value = new_opts[0]
            load_json()
    w_upload_btn.observe(_local_upload, names='value')

# Manual override inputs
w_net_mmbtu   = widgets.FloatText(value=1362090, description='Net MMBtu:', style=style, layout=layout)
w_buy_px      = widgets.FloatText(value=13.28,   description='Buy Price ($/MMBtu):', style=style, layout=layout)
w_sell_px     = widgets.FloatText(value=14.20,   description='Sell Price ($/MMBtu):', style=style, layout=layout)
w_base_pnl    = widgets.FloatText(value=0,       description='Base Net P&L ($):', style=style, layout=layout)
w_total_costs = widgets.FloatText(value=5200000, description='Total Voyage+Term Costs ($):', style=style, layout=layout)

w_load_btn.on_click(load_json)

display(widgets.VBox([
    widgets.HTML('<b style="color:#58a6ff">▸ Load from JSON (preferred) or edit manually</b>'),
    widgets.HBox([w_json_file, w_load_btn]),
    widgets.HTML('<b style="color:#8b949e;font-size:11px">▸ Or upload a JSON file from your computer</b>'),
    widgets.HBox([w_upload_btn]),
    upload_out,
    load_out,
    widgets.HTML('<b style="color:#8b949e;font-size:11px">▸ Manual overrides (auto-filled on JSON load)</b>'),
    w_net_mmbtu, w_buy_px, w_sell_px, w_base_pnl, w_total_costs,
], layout=widgets.Layout(padding='12px', border='1px solid #30363d', border_radius='8px')))

if json_opts[0] != 'No JSON found — enter manually':
    load_json()


---
## 🏦 Section 2 — Credit & Counterparty Check

In [ ]:
# ── CELL 3: Credit check ─────────────────────────────────────────────────────
w_cpty_name   = widgets.Text(value='QatarEnergy', description='Counterparty:', style=style, layout=layout)
w_cpty_rating = widgets.Dropdown(
    options=['AAA','AA+','AA','AA-','A+','A','A-',
             'BBB+','BBB','BBB-','BB+','BB','B','CCC','NR'],
    value='AA', description='Credit Rating:', style=style, layout=layout
)
w_credit_limit = widgets.FloatText(value=100000000, description='Credit Limit ($):', style=style, layout=layout)
w_existing_ap  = widgets.FloatText(value=45200000,  description='Existing AP ($):', style=style, layout=layout)
w_existing_ar  = widgets.FloatText(value=0,         description='Existing AR ($):', style=style, layout=layout)
w_collateral   = widgets.FloatText(value=0,         description='Collateral Held ($):', style=style, layout=layout)
w_country_risk = widgets.Dropdown(
    options=['Low','Medium','High','Sanctioned'],
    value='Low', description='Country Risk:', style=style, layout=layout
)

credit_out = widgets.Output()

def run_credit_check(_=None):
    deal_value    = w_net_mmbtu.value * w_buy_px.value
    new_ap        = w_existing_ap.value + deal_value
    net_exposure  = max(0, new_ap - w_existing_ar.value - w_collateral.value)
    limit         = w_credit_limit.value
    utilisation   = (net_exposure / limit * 100) if limit > 0 else 0
    headroom      = max(0, limit - net_exposure)
    rating        = w_cpty_rating.value
    country       = w_country_risk.value

    rating_limits = {
        'AAA': 200e6,'AA+': 150e6,'AA': 100e6,'AA-': 80e6,
        'A+':  60e6, 'A':  50e6,  'A-': 40e6,
        'BBB+':30e6,'BBB': 20e6,'BBB-':15e6,
        'BB+': 10e6,'BB':  5e6,  'B':   1e6,'CCC':0,'NR':0
    }
    policy_max = rating_limits.get(rating, 0)
    if country == 'High':       policy_max *= 0.5
    if country == 'Sanctioned': policy_max  = 0

    policy_ok = net_exposure <= policy_max
    limit_ok  = utilisation < 100
    country_ok = country != 'Sanctioned'

    overall = '✅ APPROVED' if (policy_ok and limit_ok and country_ok) else '❌ BLOCKED'

    with credit_out:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
        fig.patch.set_facecolor(C['bg'])

        # Gauge 1: Limit utilisation
        ax = axes[0]; ax.set_facecolor(C['card'])
        color = C['gain'] if utilisation < 80 else C['warn'] if utilisation < 100 else C['loss']
        ax.barh([''], [100], color='#21262d', height=0.4)
        ax.barh([''], [min(utilisation,100)], color=color, height=0.4)
        ax.set_xlim(0, 110); ax.set_yticks([])
        ax.axvline(80,  color=C['warn'], lw=1.2, ls='--', alpha=0.7)
        ax.axvline(100, color=C['loss'], lw=1.2, ls='--', alpha=0.7)
        ax.set_xlabel('% of Limit', fontsize=9)
        ax.set_title(f'Limit Utilisation: {utilisation:.1f}%', fontsize=10)
        ax.text(min(utilisation,100)/2, 0, f'{utilisation:.1f}%',
                ha='center', va='center', fontsize=12,
                fontweight='bold', color=C['bg'])
        ax.grid(False)

        # Bar 2: Exposure breakdown
        ax2 = axes[1]; ax2.set_facecolor(C['card'])
        cats   = ['Existing AP','This Deal','AR Offset','Collateral','Net Exposure','Limit']
        vals   = [w_existing_ap.value, deal_value,
                  -w_existing_ar.value, -w_collateral.value,
                  net_exposure, limit]
        cols2  = [C['warn'],C['loss'],C['gain'],C['gain'],
                  C['loss'] if not limit_ok else C['blue'], C['blue']]
        ax2.barh(cats, [v/1e6 for v in vals], color=cols2,
                 height=0.55, alpha=0.85)
        ax2.axvline(0, color=C['border'], lw=0.8)
        ax2.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v,_: f'${v:.0f}M'))
        ax2.set_title('Exposure vs Limit', fontsize=10)
        ax2.grid(True, axis='x', alpha=0.3)

        # Text panel 3
        ax3 = axes[2]; ax3.set_facecolor(C['card'])
        ax3.set_xticks([]); ax3.set_yticks([])
        for sp in ax3.spines.values(): sp.set_edgecolor(C['border'])
        checks = [
            ('Counterparty',   w_cpty_name.value,  True),
            ('Rating',         rating,              True),
            ('Policy Max',     f'${policy_max/1e6:.0f}M', policy_ok),
            ('Net Exposure',   f'${net_exposure/1e6:.1f}M', limit_ok),
            ('Headroom',       f'${headroom/1e6:.1f}M',   headroom > 0),
            ('Country Risk',   country,             country_ok),
            ('OVERALL',        overall,             '✅' in overall),
        ]
        for i, (lbl, val, ok) in enumerate(checks):
            ypos = 0.90 - i * 0.12
            col  = C['gain'] if ok else C['loss']
            ax3.text(0.05, ypos, lbl + ':', transform=ax3.transAxes,
                     fontsize=8.5, color=C['muted'])
            ax3.text(0.55, ypos, str(val), transform=ax3.transAxes,
                     fontsize=8.5, color=col, fontweight='bold')
        ax3.set_title('Credit Decision', fontsize=10)

        fig.suptitle(f'Credit Check — {w_cpty_name.value} — {overall}',
                     fontsize=12, fontweight='bold',
                     color=C['gain'] if '✅' in overall else C['loss'])
        plt.tight_layout()
        plt.show()
        plt.close(fig)

        print(f"  Approved Credit Limit:  ${limit:>16,.0f}")
        print(f"  Deal AP Exposure:        ${deal_value:>16,.0f}")
        print(f"  Net Exposure Post-Deal:  ${net_exposure:>16,.0f}")
        print(f"  Utilisation:             {utilisation:>15.1f}%")
        print(f"  Policy Max ({rating}):      ${policy_max:>16,.0f}")
        print(f"  DECISION: {overall}")

w_credit_btn = widgets.Button(
    description='▶  Run Credit Check',
    button_style='warning',
    layout=widgets.Layout(width='200px', height='38px')
)
w_credit_btn.on_click(run_credit_check)

display(widgets.VBox([
    widgets.HTML('<b style="color:#d29922;font-size:13px">▸ Counterparty Details</b>'),
    w_cpty_name, w_cpty_rating, w_country_risk,
    widgets.HTML('<b style="color:#d29922;font-size:13px">▸ Exposure Inputs</b>'),
    w_credit_limit, w_existing_ap, w_existing_ar, w_collateral,
    w_credit_btn, credit_out
], layout=widgets.Layout(padding='12px', border='1px solid #30363d', border_radius='8px')))


---
## 📐 Section 3 — IFRS 9 Hedge Accounting (OCI vs P&L)

In [ ]:
# ── CELL 4: IFRS 9 hedge accounting ──────────────────────────────────────────
w_hedge_type = widgets.Dropdown(
    options=['Cash Flow Hedge','Fair Value Hedge','No Hedge'],
    value='Cash Flow Hedge',
    description='IFRS 9 Hedge Type:', style=style, layout=layout
)
w_hedge_ratio_ifrs = widgets.FloatSlider(
    value=80.0, min=0.0, max=100.0, step=5.0,
    description='Hedge Ratio (%):', style=style, layout=layout,
    readout_format='.0f'
)
w_hedge_fixed_px = widgets.FloatText(
    value=13.75, description='Hedge Fixed Price ($/MMBtu):', style=style, layout=layout
)
w_current_market_px = widgets.FloatText(
    value=14.05, description='Current Market Price ($/MMBtu):', style=style, layout=layout
)
w_oci_opening = widgets.FloatText(
    value=0.0, description='Opening OCI Balance ($):', style=style, layout=layout
)
w_effectiveness = widgets.Dropdown(
    options=['Qualitative (same index/tenor)','Quantitative (regression)'],
    value='Qualitative (same index/tenor)',
    description='Effectiveness Method:', style=style, layout=layout
)

ifrs_out = widgets.Output()

def run_ifrs9(*_):
    net        = w_net_mmbtu.value
    sell_px    = w_sell_px.value
    hr         = w_hedge_ratio_ifrs.value / 100
    hedge_vol  = net * hr
    unhg_vol   = net * (1 - hr)
    fixed_px   = w_hedge_fixed_px.value
    mkt_px     = w_current_market_px.value
    oci_open   = w_oci_opening.value
    htype      = w_hedge_type.value

    # MTM of hedge instrument
    hedge_mtm  = (fixed_px - mkt_px) * hedge_vol   # gain if fixed > market

    # Physical MTM (unhedged portion only hits P&L)
    phys_mtm   = (mkt_px - sell_px) * net           # gain if market > contracted

    if htype == 'Cash Flow Hedge':
        # Hedged portion → OCI; unhedged → P&L
        to_oci  = (mkt_px - sell_px) * hedge_vol
        to_pnl  = (mkt_px - sell_px) * unhg_vol
        oci_close = oci_open + to_oci
        pnl_impact = to_pnl
        # When cargo delivers: OCI reclassified to P&L
        oci_reclassify_on_delivery = oci_close
    elif htype == 'Fair Value Hedge':
        # Both hedged item and derivative MTM through P&L — offset
        to_oci  = 0
        to_pnl  = phys_mtm + hedge_mtm
        oci_close = 0
        pnl_impact = to_pnl
        oci_reclassify_on_delivery = 0
    else:
        to_oci  = 0
        to_pnl  = phys_mtm
        oci_close = 0
        pnl_impact = to_pnl
        oci_reclassify_on_delivery = 0

    effectiveness_ok = abs(hr) > 0.8 or htype == 'No Hedge'

    with ifrs_out:
        clear_output(wait=True)
        fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4.5))
        fig.patch.set_facecolor(C['bg'])

        # Chart 1: P&L vs OCI split
        ax1.set_facecolor(C['card'])
        cats_  = ['To P&L\n(Unrealised)', 'To OCI\n(Deferred)', 'OCI on\nDelivery']
        vals_  = [to_pnl/1e6, to_oci/1e6, oci_reclassify_on_delivery/1e6]
        cols_  = [C['gain'] if v >= 0 else C['loss'] for v in vals_]
        ax1.bar(cats_, vals_, color=cols_, alpha=0.85,
                edgecolor='#21262d', linewidth=0.5)
        for i, v in enumerate(vals_):
            ax1.text(i, v + max(abs(x) for x in vals_)*0.03 if v >= 0
                     else v - max(abs(x) for x in vals_)*0.06,
                     f'${v:+.2f}M', ha='center', fontsize=9,
                     fontweight='bold', color=C['text'])
        ax1.axhline(0, color=C['border'], lw=0.8, ls='--')
        ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v,_: f'${v:.2f}M'))
        ax1.set_title(f'IFRS 9 — {htype}', fontsize=10)
        ax1.grid(True, axis='y', alpha=0.3)

        # Chart 2: Volume split hedged/unhedged
        ax2.set_facecolor(C['card'])
        ax2.pie([hedge_vol, unhg_vol],
                labels=[f'Hedged\n{hr*100:.0f}%\n{hedge_vol/1e6:.2f}M MMBtu',
                        f'Unhedged\n{(1-hr)*100:.0f}%\n{unhg_vol/1e6:.2f}M MMBtu'],
                colors=[C['blue'], C['warn']],
                startangle=90, autopct='',
                wedgeprops={'edgecolor': C['bg'], 'linewidth': 2},
                textprops={'color': C['text'], 'fontsize': 9})
        ax2.set_title('Hedge Volume Split', fontsize=10)

        # Chart 3: Journal entries visualisation
        ax3.set_facecolor(C['card'])
        ax3.set_xticks([]); ax3.set_yticks([])
        for sp in ax3.spines.values(): sp.set_edgecolor(C['border'])

        if htype == 'Cash Flow Hedge':
            entries = [
                ('Period-end (MTM on hedge):',''),
                (f'  DR: Derivative Asset', f'${abs(hedge_mtm):,.0f}'),
                (f'  CR: OCI — CF Hedge Reserve', f'${abs(hedge_mtm):,.0f}'),
                ('',''),
                ('On cargo delivery (reclassify):',''),
                (f'  DR: OCI — CF Hedge Reserve', f'${abs(oci_close):,.0f}'),
                (f'  CR: LNG Revenue (P&L)', f'${abs(oci_close):,.0f}'),
            ]
        elif htype == 'Fair Value Hedge':
            entries = [
                ('Period-end (both through P&L):',''),
                (f'  DR/CR: Hedged Item', f'${abs(phys_mtm):,.0f}'),
                (f'  DR/CR: P&L', f'${abs(phys_mtm):,.0f}'),
                (f'  DR/CR: Derivative', f'${abs(hedge_mtm):,.0f}'),
                (f'  DR/CR: P&L', f'${abs(hedge_mtm):,.0f}'),
            ]
        else:
            entries = [
                ('No hedge — MTM direct to P&L:',''),
                (f'  DR/CR: MTM Asset/Liability', f'${abs(phys_mtm):,.0f}'),
                (f'  DR/CR: Unrealised MTM P&L', f'${abs(phys_mtm):,.0f}'),
            ]

        for i, (lbl, val) in enumerate(entries):
            ypos = 0.95 - i * 0.12
            c1 = C['blue'] if lbl.startswith('  DR') else                  C['warn'] if lbl.startswith('  CR') else C['muted']
            ax3.text(0.02, ypos, lbl, transform=ax3.transAxes,
                     fontsize=7.5, color=c1)
            if val:
                ax3.text(0.75, ypos, val, transform=ax3.transAxes,
                         fontsize=7.5, color=C['text'], ha='right')
        ax3.set_title('Indicative Journal Entries', fontsize=10)

        eff_str = '✅ Effective' if effectiveness_ok else '⚠️  Review Required'
        fig.suptitle(f'IFRS 9 Hedge Accounting — {eff_str}',
                     fontsize=12, fontweight='bold',
                     color=C['gain'] if effectiveness_ok else C['warn'])
        plt.tight_layout()
        plt.show()
        plt.close(fig)

        print(f"  Hedge Type:           {htype}")
        print(f"  Hedge Ratio:          {hr*100:.0f}%")
        print(f"  Hedged Volume:        {hedge_vol:,.0f} MMBtu")
        print(f"  Hedge MTM:            ${hedge_mtm:+,.0f}")
        print(f"  To P&L (period-end):  ${to_pnl:+,.0f}")
        print(f"  To OCI (deferred):    ${to_oci:+,.0f}")
        print(f"  OCI on delivery:      ${oci_reclassify_on_delivery:+,.0f}")
        print(f"  Effectiveness:        {eff_str}")

w_ifrs_btn = widgets.Button(
    description='▶  Calculate IFRS 9',
    button_style='primary',
    layout=widgets.Layout(width='200px', height='38px')
)
w_ifrs_btn.on_click(run_ifrs9)

for w in [w_hedge_type, w_hedge_ratio_ifrs, w_hedge_fixed_px,
          w_current_market_px]:
    w.observe(run_ifrs9, names='value')

display(widgets.VBox([
    widgets.HTML('<b style="color:#bc8cff;font-size:13px">▸ IFRS 9 Configuration</b>'),
    w_hedge_type, w_effectiveness, w_hedge_ratio_ifrs,
    w_hedge_fixed_px, w_current_market_px, w_oci_opening,
    w_ifrs_btn, ifrs_out
], layout=widgets.Layout(padding='12px', border='1px solid #30363d', border_radius='8px')))


---
## 🌍 Section 4 — Multi-Destination Netback Comparison

In [ ]:
# ── CELL 5: Multi-destination netback ────────────────────────────────────────
destinations = {
    'Japan (Sodegaura)': {
        'market_px': 14.20, 'voyage_days': 20,
        'freight_mmbtu': 2.45, 'canal': 0,
        'port_fix': 270000, 'regas': 0.40,
        'insurance': 350000,
    },
    'South Korea (Incheon)': {
        'market_px': 14.10, 'voyage_days': 19,
        'freight_mmbtu': 2.40, 'canal': 0,
        'port_fix': 250000, 'regas': 0.38,
        'insurance': 340000,
    },
    'UK (Isle of Grain)': {
        'market_px': 16.50, 'voyage_days': 18,
        'freight_mmbtu': 3.20, 'canal': 450000,
        'port_fix': 270000, 'regas': 0.35,
        'insurance': 350000,
    },
    'Spain (Barcelona)': {
        'market_px': 15.80, 'voyage_days': 14,
        'freight_mmbtu': 2.80, 'canal': 350000,
        'port_fix': 240000, 'regas': 0.30,
        'insurance': 330000,
    },
    'India (Dahej)': {
        'market_px': 13.90, 'voyage_days': 9,
        'freight_mmbtu': 2.00, 'canal': 0,
        'port_fix': 200000, 'regas': 0.25,
        'insurance': 280000,
    },
    'Turkey (Marmara)': {
        'market_px': 15.20, 'voyage_days': 12,
        'freight_mmbtu': 2.50, 'canal': 350000,
        'port_fix': 220000, 'regas': 0.32,
        'insurance': 310000,
    },
}

dest_widgets = {}
for dname, dvals in destinations.items():
    dest_widgets[dname] = {
        'active': widgets.Checkbox(value=True, description=dname,
                                    layout=widgets.Layout(width='220px')),
        'market_px': widgets.FloatText(value=dvals['market_px'],
                                        description='Market Px:', style={'description_width':'90px'},
                                        layout=widgets.Layout(width='200px')),
        'freight':   widgets.FloatText(value=dvals['freight_mmbtu'],
                                        description='Freight:', style={'description_width':'90px'},
                                        layout=widgets.Layout(width='200px')),
        'canal':     widgets.FloatText(value=dvals['canal'],
                                        description='Canal ($):', style={'description_width':'90px'},
                                        layout=widgets.Layout(width='200px')),
        '_regas':    widgets.FloatText(value=dvals['regas'],
                                        description='Regas:', style={'description_width':'90px'},
                                        layout=widgets.Layout(width='200px')),
        '_port_fix': widgets.FloatText(value=dvals['port_fix'],
                                        description='Port ($):', style={'description_width':'90px'},
                                        layout=widgets.Layout(width='200px')),
        '_voyage_d': widgets.IntText(value=dvals['voyage_days'],
                                      description='Days:', style={'description_width':'90px'},
                                      layout=widgets.Layout(width='200px')),
    }

dest_panels = []
for dname, dw in dest_widgets.items():
    panel = widgets.VBox([
        dw['active'],
        dw['market_px'], dw['freight'],
        dw['canal'], dw['_regas'],
        dw['_port_fix'], dw['_voyage_d'],
    ], layout=widgets.Layout(padding='8px', border='1px solid #30363d',
                              border_radius='6px', width='230px'))
    dest_panels.append(panel)

netback_out = widgets.Output()

def run_netback(_=None):
    net      = w_net_mmbtu.value
    buy_cost = net * w_buy_px.value
    results  = []

    for dname, dw in dest_widgets.items():
        if not dw['active'].value:
            continue
        mkt   = dw['market_px'].value
        frg   = dw['freight'].value
        cnal  = dw['canal'].value
        rgs   = dw['_regas'].value
        port  = dw['_port_fix'].value
        days  = dw['_voyage_d'].value
        ins   = 350000

        bog_adj = net * (1 - 0.001 * days)
        revenue = bog_adj * mkt
        shipping_var = frg * bog_adj
        shipping_fix = cnal + port + ins
        regas_cost   = rgs * bog_adj
        total_costs  = buy_cost + shipping_var + shipping_fix + regas_cost
        net_pnl      = revenue - total_costs
        netback      = net_pnl / bog_adj if bog_adj > 0 else 0

        results.append({
            'Destination':  dname,
            'Market Px':    mkt,
            'Freight/MMBtu':frg,
            'Net MMBtu':    bog_adj,
            'Revenue':      revenue,
            'Total Costs':  total_costs,
            'Net P&L':      net_pnl,
            'Netback':      netback,
            'Voyage Days':  days,
        })

    if not results:
        with netback_out:
            clear_output(wait=True)
            print("No destinations selected.")
        return

    df = pd.DataFrame(results).sort_values('Netback', ascending=False)
    df['Rank'] = range(1, len(df)+1)

    with netback_out:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        fig.patch.set_facecolor(C['bg'])

        # Chart 1: Net P&L by destination
        ax1.set_facecolor(C['card'])
        colors_ = [C['gain'] if v >= 0 else C['loss']
                   for v in df['Net P&L']]
        bars = ax1.bar(df['Destination'], df['Net P&L']/1e6,
                       color=colors_, alpha=0.85,
                       edgecolor='#21262d', linewidth=0.5)
        # Highlight best
        best_idx = df['Net P&L'].idxmax()
        bars[df.index.get_loc(best_idx)].set_edgecolor(C['warn'])
        bars[df.index.get_loc(best_idx)].set_linewidth(2.5)

        for bar, val in zip(bars, df['Net P&L']):
            ax1.text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + abs(df['Net P&L']).max()*0.03,
                     f'${val/1e6:+.2f}M',
                     ha='center', fontsize=8.5,
                     fontweight='bold', color=C['text'])
        ax1.axhline(0, color=C['border'], lw=0.8, ls='--')
        ax1.set_xticklabels(df['Destination'], rotation=18, ha='right', fontsize=8)
        ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v,_: f'${v:.1f}M'))
        ax1.set_title('Net P&L by Destination', fontsize=11)
        ax1.grid(True, axis='y', alpha=0.3)

        # Chart 2: Stacked cost + revenue
        ax2.set_facecolor(C['card'])
        x = np.arange(len(df))
        ax2.bar(x, df['Revenue']/1e6, color=C['blue'],
                alpha=0.6, label='Revenue', width=0.6)
        ax2.bar(x, -df['Total Costs']/1e6, color=C['loss'],
                alpha=0.6, label='Total Costs', width=0.6)
        ax2.axhline(0, color=C['border'], lw=0.8, ls='--')
        ax2.set_xticks(x)
        ax2.set_xticklabels(df['Destination'], rotation=18, ha='right', fontsize=8)
        ax2.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v,_: f'${v:.0f}M'))
        ax2.set_title('Revenue vs Costs', fontsize=11)
        ax2.legend(fontsize=8)
        ax2.grid(True, axis='y', alpha=0.3)

        best = df.iloc[0]
        fig.suptitle(
            f'Netback Analysis — Best Destination: {best["Destination"]} '
            f'(Netback: ${best["Netback"]:+.3f}/MMBtu)',
            fontsize=11, fontweight='bold', color=C['warn']
        )
        plt.tight_layout()
        plt.show()
        plt.close(fig)

        # Table
        display_df = df[['Rank','Destination','Market Px','Freight/MMBtu',
                         'Voyage Days','Net P&L','Netback']].copy()
        display_df['Net P&L']  = display_df['Net P&L'].map('${:+,.0f}'.format)
        display_df['Netback']  = display_df['Netback'].map('${:+.4f}'.format)
        display_df['Market Px']= display_df['Market Px'].map('${:.2f}'.format)
        display_df['Freight/MMBtu'] = display_df['Freight/MMBtu'].map('${:.2f}'.format)
        print()
        display(display_df.set_index('Rank'))

w_netback_btn = widgets.Button(
    description='▶  Run Netback Analysis',
    button_style='success',
    layout=widgets.Layout(width='220px', height='38px')
)
w_netback_btn.on_click(run_netback)

grid_rows = []
row = []
for i, p in enumerate(dest_panels):
    row.append(p)
    if len(row) == 3:
        grid_rows.append(widgets.HBox(row, layout=widgets.Layout(gap='8px')))
        row = []
if row:
    grid_rows.append(widgets.HBox(row, layout=widgets.Layout(gap='8px')))

display(widgets.VBox([
    widgets.HTML('<b style="color:#3fb950;font-size:13px">▸ Configure Destinations (edit market prices & costs)</b>'),
    *grid_rows,
    widgets.HBox([w_netback_btn]),
    netback_out
], layout=widgets.Layout(padding='12px', border='1px solid #30363d', border_radius='8px')))


---
## 🚢 Section 5 — Vessel Selection & Laycan Scheduling

In [ ]:
# ── CELL 6: Vessel scheduling ────────────────────────────────────────────────
vessels_db = {
    'Pacific Pearl (155,000 m³)':  {'cap': 155000, 'bog': 0.09, 'day_rate': 115000, 'curr_pos': 'Singapore', 'avail': 10},
    'Arctic Voyager (125,000 m³)': {'cap': 125000, 'bog': 0.12, 'day_rate':  95000, 'curr_pos': 'Yokohama',  'avail': 12},
    'LNG Harmony (174,000 m³)':    {'cap': 174000, 'bog': 0.08, 'day_rate': 135000, 'curr_pos': 'Rotterdam', 'avail': 15},
    'Qatar Max I (266,000 m³)':    {'cap': 266000, 'bog': 0.08, 'day_rate': 185000, 'curr_pos': 'Ras Laffan','avail': 3},
    'Spot Charter (TBN)':          {'cap': 155000, 'bog': 0.10, 'day_rate': 120000, 'curr_pos': 'TBN',       'avail': 14},
}

w_vessel = widgets.Dropdown(
    options=list(vessels_db.keys()),
    value='Pacific Pearl (155,000 m³)',
    description='Vessel:', style=style, layout=layout
)
w_laycan_start_days = widgets.IntSlider(
    value=18, min=5, max=60, step=1,
    description='Laycan Start (days from today):', style=style, layout=layout
)
w_laycan_window = widgets.IntSlider(
    value=3, min=2, max=7, step=1,
    description='Laycan Window (days):', style=style, layout=layout
)
w_load_port_name = widgets.Text(
    value='Ras Laffan, Qatar',
    description='Load Port:', style=style, layout=layout
)
w_disch_port_name = widgets.Text(
    value='Isle of Grain, UK',
    description='Discharge Port:', style=style, layout=layout
)
w_days_to_load_port = widgets.IntSlider(
    value=4, min=1, max=15, step=1,
    description='Ballast Days to Load Port:', style=style, layout=layout
)
w_allowed_laytime = widgets.FloatText(
    value=36.0, description='Allowed Laytime (hours):', style=style, layout=layout
)

vessel_out = widgets.Output()

def run_vessel(*_):
    vsel  = w_vessel.value
    vdata = vessels_db[vsel]
    today = date.today()
    laycan_start = today + timedelta(days=w_laycan_start_days.value)
    laycan_end   = laycan_start + timedelta(days=w_laycan_window.value)
    vessel_avail = today + timedelta(days=vdata['avail'])
    ballast_arr  = vessel_avail + timedelta(days=w_days_to_load_port.value)

    can_make_laycan = ballast_arr <= laycan_end
    cargo_vol  = w_net_mmbtu.value
    bog_rate   = vdata['bog'] / 100
    voyage_d   = w_voyage_days.value if hasattr(w_voyage_days, 'value') else 20

    # Costs
    freight_cost  = cargo_vol * w_freight_rate.value if hasattr(w_freight_rate,'value') else cargo_vol * 2.45
    day_rate_cost = vdata['day_rate'] * (voyage_d + w_days_to_load_port.value)
    bog_cost      = cargo_vol * bog_rate * voyage_d
    heel_days     = w_days_to_load_port.value
    heel_bog      = cargo_vol * 0.001 * heel_days

    with vessel_out:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        fig.patch.set_facecolor(C['bg'])

        # Chart 1: Timeline Gantt
        ax1 = axes[0]; ax1.set_facecolor(C['card'])
        events = [
            ('Vessel Available',  0,              vdata['avail'],        C['muted']),
            ('Ballast to Load',   vdata['avail'], w_days_to_load_port.value, C['warn']),
            ('Laycan Window',     w_laycan_start_days.value, w_laycan_window.value, C['blue']),
            ('Loading',           w_laycan_start_days.value + 1, 1,     C['gain']),
            ('Voyage',            w_laycan_start_days.value + 1, voyage_d, C['purple']),
        ]
        yticks = []
        for i, (name, start, dur, col) in enumerate(events):
            ax1.barh(i, dur, left=start, color=col,
                     alpha=0.8, height=0.6,
                     edgecolor='#21262d', linewidth=0.5)
            yticks.append(name)
        ax1.set_yticks(range(len(events)))
        ax1.set_yticklabels(yticks, fontsize=8)
        ax1.set_xlabel('Days from Today', fontsize=8)
        ax1.set_title('Voyage Timeline', fontsize=10)
        ax1.grid(True, axis='x', alpha=0.3)
        flag = '✅' if can_make_laycan else '❌'
        ax1.set_title(f'Timeline {flag}', fontsize=10)

        # Chart 2: Vessel comparison
        ax2 = axes[1]; ax2.set_facecolor(C['card'])
        v_names   = list(vessels_db.keys())
        v_rates   = [v['day_rate']/1000 for v in vessels_db.values()]
        v_colors_ = [C['gain'] if k == vsel else C['muted'] for k in vessels_db]
        bars2 = ax2.barh(v_names, v_rates, color=v_colors_,
                          alpha=0.85, height=0.55)
        ax2.set_xlabel('Day Rate ($k/day)', fontsize=8)
        ax2.set_yticklabels(v_names, fontsize=7.5)
        ax2.set_title('Vessel Day Rates', fontsize=10)
        ax2.grid(True, axis='x', alpha=0.3)

        # Text panel
        ax3 = axes[2]; ax3.set_facecolor(C['card'])
        ax3.set_xticks([]); ax3.set_yticks([])
        for sp in ax3.spines.values(): sp.set_edgecolor(C['border'])

        info = [
            ('Vessel',          vsel.split('(')[0].strip()),
            ('Capacity (m³)',   f"{vdata['cap']:,}"),
            ('BOG Rate',        f"{vdata['bog']:.2f}%/day"),
            ('Current Position',vdata['curr_pos']),
            ('Available',       str(vessel_avail)),
            ('Laycan Start',    str(laycan_start)),
            ('Laycan End',      str(laycan_end)),
            ('Est Arrive Load', str(ballast_arr)),
            ('Can Make Laycan', '✅ YES' if can_make_laycan else '❌ NO'),
            ('Voyage BOG Loss', f'{bog_cost:,.0f} MMBtu'),
            ('Day Rate Cost',   f'${day_rate_cost:,.0f}'),
        ]
        for i, (lbl, val) in enumerate(info):
            ypos = 0.95 - i * 0.085
            ok_  = '✅' in val or '❌' not in val
            col_ = C['gain'] if '✅' in val else                    C['loss'] if '❌' in val else C['text']
            ax3.text(0.02, ypos, lbl + ':', transform=ax3.transAxes,
                     fontsize=8, color=C['muted'])
            ax3.text(0.52, ypos, val, transform=ax3.transAxes,
                     fontsize=8, color=col_, fontweight='bold')
        ax3.set_title('Vessel Assignment', fontsize=10)

        fig.suptitle(f'Vessel: {vsel}',
                     fontsize=11, fontweight='bold', color=C['text'])
        plt.tight_layout()
        plt.show()
        plt.close(fig)

        print(f"  Vessel:           {vsel}")
        print(f"  Laycan:           {laycan_start} – {laycan_end}")
        print(f"  Vessel Available: {vessel_avail}")
        print(f"  Est Arrive Load:  {ballast_arr}")
        print(f"  Can Make Laycan:  {'✅ YES' if can_make_laycan else '❌ NO'}")
        print(f"  Allowed Laytime:  {w_allowed_laytime.value:.1f} hours")

w_freight_rate = widgets.FloatText(
    value=2.45, description='Freight Rate ($/MMBtu):', style=style, layout=layout
)
w_voyage_days  = widgets.IntSlider(
    value=20, min=5, max=35, step=1,
    description='Voyage Days:', style=style, layout=layout
)

w_vessel_btn = widgets.Button(
    description='▶  Assign Vessel',
    button_style='primary',
    layout=widgets.Layout(width='180px', height='38px')
)
w_vessel_btn.on_click(run_vessel)
for w in [w_vessel, w_laycan_start_days, w_laycan_window, w_voyage_days]:
    w.observe(run_vessel, names='value')

display(widgets.VBox([
    widgets.HTML('<b style="color:#39d353;font-size:13px">▸ Vessel & Scheduling Inputs</b>'),
    w_vessel, w_laycan_start_days, w_laycan_window,
    w_load_port_name, w_disch_port_name,
    w_days_to_load_port, w_voyage_days,
    w_freight_rate, w_allowed_laytime,
    w_vessel_btn, vessel_out
], layout=widgets.Layout(padding='12px', border='1px solid #30363d', border_radius='8px')))
run_vessel()


---
## ✅ Section 6 — Deal Approval Workflow

In [ ]:
# ── CELL 7: Approval workflow simulation ─────────────────────────────────────
w_trader_name = widgets.Text(
    value='Sarah Chen', description='Trader:', style=style, layout=layout
)
w_book = widgets.Text(
    value='ASIA_ST_SPOT', description='Book:', style=style, layout=layout
)
w_deal_notes = widgets.Textarea(
    value='Spot cargo, diverted to Europe for TTF arbitrage.',
    description='Deal Rationale:', style=style,
    layout=widgets.Layout(width='500px', height='70px')
)
w_mo_approve  = widgets.ToggleButton(value=False, description='Middle Office ✓',
    button_style='', layout=widgets.Layout(width='180px'))
w_risk_approve = widgets.ToggleButton(value=False, description='Risk ✓',
    button_style='', layout=widgets.Layout(width='180px'))
w_credit_approve = widgets.ToggleButton(value=False, description='Credit ✓',
    button_style='', layout=widgets.Layout(width='180px'))
w_hot_approve = widgets.ToggleButton(value=False, description='Head of Trading ✓',
    button_style='', layout=widgets.Layout(width='180px'))

approval_out = widgets.Output()

def update_approval(*_):
    deal_val = w_net_mmbtu.value * w_sell_px.value
    need_mo  = deal_val > 5e6
    need_hot = deal_val > 20e6
    need_risk = True
    need_credit = True

    checks = {
        'Middle Office':   (need_mo,  w_mo_approve.value),
        'Risk':            (need_risk, w_risk_approve.value),
        'Credit':          (need_credit, w_credit_approve.value),
        'Head of Trading': (need_hot, w_hot_approve.value),
    }

    required_met = all(
        (not required or approved)
        for _, (required, approved) in checks.items()
    )

    for btn, (required, approved) in zip(
        [w_mo_approve, w_risk_approve, w_credit_approve, w_hot_approve],
        checks.values()
    ):
        if approved:
            btn.button_style = 'success'
        elif not checks[btn.description.replace(' ✓','')][0]:
            btn.button_style = 'info'
        else:
            btn.button_style = 'danger'

    with approval_out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(12, 3))
        fig.patch.set_facecolor(C['bg'])
        ax.set_facecolor(C['card'])
        ax.set_xlim(-0.5, 4.5); ax.set_ylim(0, 2)
        ax.set_xticks([]); ax.set_yticks([])

        stages = ['Trader\nSubmits', 'Middle\nOffice', 'Risk\nTeam',
                  'Credit\nTeam', 'Head of\nTrading']
        x_pos  = [0, 1, 2, 3, 4]
        status = [
            True,
            w_mo_approve.value   if need_mo   else None,
            w_risk_approve.value if need_risk  else None,
            w_credit_approve.value if need_credit else None,
            w_hot_approve.value  if need_hot   else None,
        ]

        for i, (stage, st) in enumerate(zip(stages, status)):
            col = C['gain'] if st is True else                   C['muted'] if st is None else                   C['loss']
            circ = plt.Circle((x_pos[i], 1), 0.3, color=col,
                               alpha=0.9, zorder=3)
            ax.add_patch(circ)
            icon = '✓' if st is True else                    '○' if st is None else '○'
            ax.text(x_pos[i], 1, icon, ha='center', va='center',
                    fontsize=14, fontweight='bold',
                    color=C['bg'] if st else C['text'], zorder=4)
            ax.text(x_pos[i], 0.4, stage, ha='center',
                    fontsize=8, color=C['text'])
            if i < 4:
                ax.annotate('', xy=(x_pos[i+1]-0.3, 1),
                             xytext=(x_pos[i]+0.3, 1),
                             arrowprops=dict(arrowstyle='->', color=C['border'],
                                             lw=1.5))

        verdict = '✅ DEAL APPROVED' if required_met else '⏳ PENDING APPROVALS'
        v_color = C['gain'] if required_met else C['warn']
        ax.text(2, 1.75, verdict, ha='center', va='center',
                fontsize=13, fontweight='bold', color=v_color)
        ax.text(2, 1.55,
                f'Deal Value: ${deal_val/1e6:.1f}M  |  Trader: {w_trader_name.value}  |  Book: {w_book.value}',
                ha='center', va='center', fontsize=8.5, color=C['muted'])
        plt.tight_layout()
        plt.show()
        plt.close(fig)

        print(f"  Deal Value:    ${deal_val:,.0f}")
        print(f"  Requires MO:   {'Yes' if need_mo else 'No'}")
        print(f"  Requires HoT:  {'Yes' if need_hot else 'No'}")
        print(f"  Status:        {verdict}")
        if w_deal_notes.value:
            print(f"  Rationale:     {w_deal_notes.value}")

for btn in [w_mo_approve, w_risk_approve, w_credit_approve, w_hot_approve]:
    btn.observe(update_approval, names='value')

display(widgets.VBox([
    widgets.HTML('<b style="color:#58a6ff;font-size:13px">▸ Deal Details</b>'),
    w_trader_name, w_book, w_deal_notes,
    widgets.HTML('<b style="color:#58a6ff;font-size:13px">▸ Approver Actions (toggle to approve)</b>'),
    widgets.HBox([w_mo_approve, w_risk_approve,
                  w_credit_approve, w_hot_approve],
                  layout=widgets.Layout(gap='8px')),
    approval_out
], layout=widgets.Layout(padding='12px', border='1px solid #30363d', border_radius='8px')))
update_approval()


---
## 📊 Section 7 — IFRS 9 Adjusted P&L Summary
*Produces the approval-ready P&L incorporating OCI treatment.*

In [ ]:
import base64
# ── CELL 8: Approval-ready P&L + export enhanced JSON ────────────────────────
enhanced_out = widgets.Output()

def run_enhanced(_=None):
    net       = w_net_mmbtu.value
    buy_px    = w_buy_px.value
    sell_px   = w_sell_px.value
    hr        = w_hedge_ratio_ifrs.value / 100
    hedge_px  = w_hedge_fixed_px.value
    mkt_px    = w_current_market_px.value
    costs     = w_total_costs.value
    base_pnl  = w_base_pnl.value

    revenue   = net * sell_px
    cogs      = net * buy_px
    phys_mg   = revenue - cogs

    hedge_vol = net * hr
    unhg_vol  = net * (1 - hr)

    # IFRS 9 split
    to_pnl_mtm = (mkt_px - sell_px) * unhg_vol
    to_oci_mtm = (mkt_px - sell_px) * hedge_vol

    ebitda    = phys_mg - costs
    tax       = max(0, ebitda * 0.17)
    net_pnl_reported = ebitda - tax + to_pnl_mtm  # OCI excluded
    net_pnl_economic = net_pnl_reported + to_oci_mtm

    with enhanced_out:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        fig.patch.set_facecolor(C['bg'])

        # Chart 1: Reported vs Economic P&L
        ax1.set_facecolor(C['card'])
        labels = ['Revenue', 'COGS', 'Voyage+Term\nCosts',
                  'EBITDA', 'Tax',
                  'MTM\n(P&L portion)', 'NET P&L\n(Reported)',
                  'OCI\n(Deferred)', 'Economic\nP&L']
        vals_rep = [revenue, -cogs, -costs,
                    ebitda, -tax,
                    to_pnl_mtm, net_pnl_reported,
                    to_oci_mtm, net_pnl_economic]
        cols_ = [C['blue'], C['loss'], C['loss'],
                 C['blue'], C['loss'],
                 C['gain'] if to_pnl_mtm >= 0 else C['loss'],
                 C['gain'] if net_pnl_reported >= 0 else C['loss'],
                 C['purple'],
                 C['gain'] if net_pnl_economic >= 0 else C['loss']]

        bars_ = ax1.bar(range(len(labels)),
                        [v/1e6 for v in vals_rep],
                        color=cols_, alpha=0.85,
                        edgecolor='#21262d', linewidth=0.5)

        for bar, val in zip(bars_, vals_rep):
            yoff = abs(max(vals_rep, key=abs))/1e6 * 0.04
            ax1.text(bar.get_x() + bar.get_width()/2,
                     bar.get_height()/1e6 + (yoff if val >= 0 else -yoff*2.5),
                     f'${val/1e6:+.2f}M',
                     ha='center', fontsize=7.5,
                     fontweight='bold', color=C['text'])
        ax1.axhline(0, color=C['border'], lw=0.8, ls='--')
        ax1.set_xticks(range(len(labels)))
        ax1.set_xticklabels(labels, fontsize=8, rotation=15, ha='right')
        ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v,_: f'${v:.1f}M'))
        ax1.set_title('IFRS 9 Adjusted P&L', fontsize=11)
        ax1.grid(True, axis='y', alpha=0.3)

        # Chart 2: Reported vs Economic comparison
        ax2.set_facecolor(C['card'])
        comp_labels = ['Reported P&L\n(income statement)',
                       'OCI Balance\n(equity)',
                       'Economic P&L\n(total)']
        comp_vals   = [net_pnl_reported, to_oci_mtm, net_pnl_economic]
        comp_cols   = [C['gain'] if v >= 0 else C['loss'] for v in comp_vals]
        comp_cols[1] = C['purple']

        ax2.bar(comp_labels, [v/1e6 for v in comp_vals],
                color=comp_cols, alpha=0.85,
                edgecolor='#21262d', linewidth=0.5, width=0.5)
        for i, val in enumerate(comp_vals):
            ax2.text(i, val/1e6 + abs(max(comp_vals,key=abs))/1e6*0.04,
                     f'${val/1e6:+.2f}M',
                     ha='center', fontsize=11,
                     fontweight='bold', color=C['text'])
        ax2.axhline(0, color=C['border'], lw=0.8, ls='--')
        ax2.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v,_: f'${v:.2f}M'))
        ax2.set_title('Reported vs Economic P&L', fontsize=11)
        ax2.grid(True, axis='y', alpha=0.3)

        verdict = '✅ GO' if net_pnl_economic > 0 else '⚠️  MARGINAL' if net_pnl_economic > -1e6 else '❌ NO GO'
        fig.suptitle(f'Approval Summary — {verdict}  |  '
                     f'Economic P&L: ${net_pnl_economic/1e6:+.2f}M',
                     fontsize=12, fontweight='bold',
                     color=C['gain'] if '✅' in verdict else
                           C['warn'] if '⚠' in verdict else C['loss'])
        plt.tight_layout()
        plt.show()
        plt.close(fig)

        # Export enhanced JSON
        enhanced = {
            'source': 'lng_feasibility_extended.ipynb (Option A)',
            'generated_date': str(date.today()),
            'verdict': verdict.replace('✅ ','').replace('⚠️  ','').replace('❌ ',''),
            'counterparty': w_cpty_name.value,
            'trader': w_trader_name.value,
            'book': w_book.value,
            'volumes': {'net_mmbtu': net},
            'pricing': {
                'buy_px': buy_px, 'sell_px': sell_px,
                'hedge_ratio': hr, 'hedge_px': hedge_px,
            },
            'pnl_ifrs9': {
                'revenue': round(revenue),
                'cogs': round(cogs),
                'total_costs': round(costs),
                'ebitda': round(ebitda),
                'tax': round(tax),
                'mtm_to_pnl': round(to_pnl_mtm),
                'mtm_to_oci': round(to_oci_mtm),
                'net_pnl_reported': round(net_pnl_reported),
                'net_pnl_economic': round(net_pnl_economic),
            },
            'approvals': {
                'middle_office': w_mo_approve.value,
                'risk': w_risk_approve.value,
                'credit': w_credit_approve.value,
                'head_of_trading': w_hot_approve.value,
            }
        }

        fname = f'lng_deal_extended_{date.today().strftime("%Y%m%d")}.json'
        json_str = json.dumps(enhanced, indent=2)
        with open(fname, 'w') as f:
            f.write(json_str)
        print(
            f'✅  Enhanced deal JSON saved: {fname}'
        )
        print(f"   → This file is the input for lng_lifecycle.ipynb (Option B)")
        print(f"   Reported P&L:  ${net_pnl_reported/1e6:+.2f}M")
        print(f"   OCI Balance:   ${to_oci_mtm/1e6:+.2f}M")
        print(f"   Economic P&L:  ${net_pnl_economic/1e6:+.2f}M")

        # Trigger browser download
        if IN_COLAB:
            colab_files.download(fname)
        else:
            b64 = base64.b64encode(json_str.encode()).decode()
            display(HTML(
                f'<a download="{fname}" '
                f'href="data:application/json;base64,{b64}" '
                f'style="font-size:14px;color:#58a6ff;">'
                f'⬇ Click here to download {fname}</a>'
            ))

w_enhanced_btn = widgets.Button(
    description='▶  Generate Approval Summary + Export',
    button_style='success',
    layout=widgets.Layout(width='320px', height='42px')
)
w_enhanced_btn.on_click(run_enhanced)
display(w_enhanced_btn)
display(enhanced_out)
